In [47]:
import pandas as pd
from sklearn.impute import KNNImputer

# Load dataset
df = pd.read_csv("Feeding Dashboard data.csv")

# Remove rows with >80% missing medical data
threshold = 0.8 * (len(df.columns) - 2)
df_clean = df.dropna(thresh=threshold, subset=df.columns.difference(['encounterId', 'referral']))

# Cap outliers (e.g., BMI)
df_clean['bmi'] = df_clean['bmi'].clip(lower=15, upper=40)

# Impute missing values using KNN
imputer = KNNImputer(n_neighbors=5)
df_imputed = pd.DataFrame(imputer.fit_transform(df_clean.drop(columns=['encounterId', 'referral'])),
                          columns=df_clean.columns.difference(['encounterId', 'referral']))
df_imputed['encounterId'] = df_clean['encounterId'].reset_index(drop=True)
df_imputed['referral'] = df_clean['referral'].reset_index(drop=True)


# Save cleaned dataset
df_imputed.to_csv("Feeding_Dashboard_Cleaned.csv", index=False)

# Summary statistics
print(df_imputed.describe())

/tmp/ipykernel_51045/3936393158.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['bmi'] = df_clean['bmi'].clip(lower=15, upper=40)


               bmi  end_tidal_co2     feed_vol  feed_vol_adm         fio2  \
count  2369.000000    2369.000000  2369.000000   2369.000000  2369.000000   
mean      4.426715     921.297562    42.806070     36.604775   245.758993   
std       0.860000     183.260821    26.243950     10.385316    99.903361   
min       1.200000      45.000000     0.000000     21.000000    10.545499   
25%       3.909458     940.495359    28.516667     29.920635   186.892462   
50%       4.341619    1000.000000    39.076862     34.699195   240.773362   
75%       4.830000    1000.000000    49.965674     40.645017   302.422723   
max      12.596368    1941.230769   200.000000    100.000000   937.969925   

        fio2_ratio    insp_time  oxygen_flow_rate         peep          pip  \
count  2369.000000  2369.000000       2369.000000  2369.000000  2369.000000   
mean      1.102522    18.487535          7.280286    19.026406     7.694663   
std       0.244595    13.031598          2.123122     4.674753     6.

In [61]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import joblib

# Load the cleaned dataset
# Replace with the path to your cleaned CSV file
df = pd.read_csv("Feeding_Dashboard_Cleaned.csv")

# Check dataset
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Prepare features (X) and target (y)
# Drop encounterId (identifier) and referral (target)
X = df.drop(columns=['encounterId', 'referral'])
y = df['referral']

# Split into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("\nModel Evaluation:")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)
print("[[True Negative, False Positive], [False Negative, True Positive]]")

# Feature importance (coefficients)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})
feature_importance = feature_importance.sort_values(by='Coefficient', ascending=False)
print("\nFeature Importance:")
print(feature_importance)

# Save the model and scaler
joblib.dump(model, 'referral_prediction_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("\nModel saved as 'referral_prediction_model.pkl'")
print("Scaler saved as 'scaler.pkl'")

# Optional: Train a Random Forest model (uncomment to use)
"""
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_y_pred = rf_model.predict(X_test_scaled)
print("\nRandom Forest Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, rf_y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, rf_y_pred):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_y_pred))
"""

Dataset Shape: (2369, 18)
Columns: ['bmi', 'end_tidal_co2', 'feed_vol', 'feed_vol_adm', 'fio2', 'fio2_ratio', 'insp_time', 'oxygen_flow_rate', 'peep', 'pip', 'resp_rate', 'sip', 'tidal_vol', 'tidal_vol_actual', 'tidal_vol_kg', 'tidal_vol_spon', 'encounterId', 'referral']

Model Evaluation:
Accuracy: 0.7236
F1-Score: 0.5944
Confusion Matrix:
[[247  46]
 [ 85  96]]
[[True Negative, False Positive], [False Negative, True Positive]]

Feature Importance:
             Feature  Coefficient
9                pip     0.781649
7   oxygen_flow_rate     0.477631
6          insp_time     0.279847
14      tidal_vol_kg     0.192224
0                bmi     0.163575
10         resp_rate     0.134247
11               sip     0.065363
5         fio2_ratio     0.060307
8               peep     0.041014
13  tidal_vol_actual     0.022345
12         tidal_vol    -0.016921
2           feed_vol    -0.134724
1      end_tidal_co2    -0.266424
15    tidal_vol_spon    -0.306064
4               fio2    -0.505029
3 

'\nfrom sklearn.ensemble import RandomForestClassifier\nrf_model = RandomForestClassifier(n_estimators=100, random_state=42)\nrf_model.fit(X_train_scaled, y_train)\nrf_y_pred = rf_model.predict(X_test_scaled)\nprint("\nRandom Forest Evaluation:")\nprint(f"Accuracy: {accuracy_score(y_test, rf_y_pred):.4f}")\nprint(f"F1-Score: {f1_score(y_test, rf_y_pred):.4f}")\nprint("Confusion Matrix:")\nprint(confusion_matrix(y_test, rf_y_pred))\n'

In [67]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import joblib
import sys

def load_files(cleaned_csv_path, model_path, scaler_path):
    """Load cleaned dataset, model, and scaler with error handling."""
    try:
        df = pd.read_csv(cleaned_csv_path)
        model = joblib.load(model_path)
        scaler = joblib.load(scaler_path)
        return df, model, scaler
    except FileNotFoundError as e:
        print(f"Error: File not found - {e}")
        sys.exit(1)
    except Exception as e:
        print(f"Error loading files: {e}")
        sys.exit(1)

def prepare_test_data(df, test_size=0.2, random_state=42):
    """Prepare test data from cleaned dataset."""
    try:
        X = df.drop(columns=['encounterId', 'referral'])
        y = df['referral']
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        return X_test, y_test
    except KeyError as e:
        print(f"Error: Missing columns in dataset - {e}")
        sys.exit(1)
    except Exception as e:
        print(f"Error preparing test data: {e}")
        sys.exit(1)

def evaluate_model(model, scaler, X_test, y_test):
    """Evaluate model on test data."""
    try:
        X_test_scaled = scaler.transform(X_test)
        y_pred = model.predict(X_test_scaled)
        
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        conf_matrix = confusion_matrix(y_test, y_pred)
        
        print("\nTest Set Evaluation:")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print("Confusion Matrix:")
        print(conf_matrix)
        print("[[True Negative, False Positive], [False Negative, True Positive]]")
        
        return y_pred
    except Exception as e:
        print(f"Error evaluating model: {e}")
        sys.exit(1)

def predict_new_data(model, scaler, new_data, feature_columns):
    """Predict referral for new data (single row or DataFrame)."""
    try:
        if isinstance(new_data, dict):
            new_data = pd.DataFrame([new_data])
        elif not isinstance(new_data, pd.DataFrame):
            raise ValueError("new_data must be a DataFrame or dictionary")
        
        # Ensure new_data has the same features
        missing_cols = set(feature_columns) - set(new_data.columns)
        if missing_cols:
            raise ValueError(f"New data missing columns: {missing_cols}")
        
        # Keep only the required features in the correct order
        new_data = new_data[feature_columns]
        new_data_scaled = scaler.transform(new_data)
        predictions = model.predict(new_data_scaled)
        
        return predictions
    except Exception as e:
        print(f"Error predicting new data: {e}")
        return None

def main():
    # File paths
    cleaned_csv_path = "Feeding_Dashboard_Cleaned.csv"
    model_path = "referral_prediction_model.pkl"
    scaler_path = "scaler.pkl"
    
    # Load files
    print("Loading dataset, model, and scaler...")
    df, model, scaler = load_files(cleaned_csv_path, model_path, scaler_path)
    
    # Verify dataset
    print("\nDataset Info:")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    
    # Prepare test data
    print("\nPreparing test data...")
    X_test, y_test = prepare_test_data(df)
    print("Test Set Shape:", X_test.shape)
    
    # Evaluate model on test set
    print("\nEvaluating model on test set...")
    y_pred = evaluate_model(model, scaler, X_test, y_test)
    
    # Example: Predict for a single new patient (modify with actual values)
    feature_columns = df.drop(columns=['encounterId', 'referral']).columns.tolist()
    sample_new_data = {
        'end_tidal_co2': 3.9,
        'feed_vol': 1000.0,
        'feed_vol_adm': 45.7,
        'fio2': 39.0,
        'fio2_ratio': 240.6,
        'insp_time': 1.1,
        'oxygen_flow_rate': 2.5,
        'peep': 6.2,
        'pip': 19.5,
        'resp_rate': 2.4,
        'sip': 0.0,
        'tidal_vol': 320.0,
        'tidal_vol_actual': 346.5,
        'tidal_vol_kg': 6.9,
        'tidal_vol_spon': 200.0,
        'bmi': 25.0,
    }
    
    print("\nPredicting for sample new patient...")
    prediction = predict_new_data(model, scaler, sample_new_data, feature_columns)
    print("Sample Prediction (0.0 = No Referral, 1.0 = Referral):", prediction)
    
    # Optional: Predict for a new CSV file (uncomment to use)
    """
    new_data_path = "new_patients.csv"
    try:
        new_data = pd.read_csv(new_data_path)
        predictions = predict_new_data(model, scaler, new_data, feature_columns)
        print("\nPredictions for new CSV data:")
        print(predictions)
    except FileNotFoundError:
        print(f"Error: {new_data_path} not found")
    """

if __name__ == "__main__":
    main()

Loading dataset, model, and scaler...



Dataset Info:
Shape: (2369, 18)
Columns: ['bmi', 'end_tidal_co2', 'feed_vol', 'feed_vol_adm', 'fio2', 'fio2_ratio', 'insp_time', 'oxygen_flow_rate', 'peep', 'pip', 'resp_rate', 'sip', 'tidal_vol', 'tidal_vol_actual', 'tidal_vol_kg', 'tidal_vol_spon', 'encounterId', 'referral']

Preparing test data...
Test Set Shape: (474, 16)

Evaluating model on test set...

Test Set Evaluation:
Accuracy: 0.7236
F1-Score: 0.5944
Confusion Matrix:
[[247  46]
 [ 85  96]]
[[True Negative, False Positive], [False Negative, True Positive]]

Predicting for sample new patient...
Sample Prediction (0.0 = No Referral, 1.0 = Referral): [1.]
